EfficientNetB0 + Random Forest Classifier
Bài toán phân loại ảnh chim (150 lớp) sử dụng EfficientNetB0 với Transfer Learning.

Thay lớp Dense phân loại cuối bằng **GridSearchCV + RandomForestClassifier**.

## 1. Import Thư Viện

In [1]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, BatchNormalization, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_recall_fscore_support)
from google.colab import drive
import joblib

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.19.0


## 2. Load Dataset

In [2]:
drive.mount('/content/drive')

PATH_ORIGINAL_ZIP = '/content/drive/MyDrive/CS231/original_dataset.zip'
EXTRACT_DIR_ORIG    = '/content/original_dataset'

def extract_zip(zip_path, extract_path):
    if not os.path.exists(zip_path):
        print(f'LOI: Khong tim thay file {zip_path}.')
        return
    if not os.path.exists(extract_path):
        os.makedirs(extract_path, exist_ok=True)
        print(f'Dang giai nen {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_path)
        print(f'Giai nen xong: {extract_path}')
    else:
        print(f'Thu muc da ton tai: {extract_path}')

extract_zip(PATH_ORIGINAL_ZIP, EXTRACT_DIR_ORIG)

ORIG_PATH = {
    'train': os.path.join(EXTRACT_DIR_ORIG, 'train'),
    'val':   os.path.join(EXTRACT_DIR_ORIG, 'val'),
    'test':  os.path.join(EXTRACT_DIR_ORIG, 'test')
}

Mounted at /content/drive
Dang giai nen /content/drive/MyDrive/CS231/original_dataset.zip...
Giai nen xong: /content/original_dataset


## 3. Data Generators

In [3]:
datagen = ImageDataGenerator()

train_gen = datagen.flow_from_directory(ORIG_PATH['train'], target_size=(224, 224),
                                        batch_size=32, class_mode='categorical',
                                        shuffle=False)
val_gen   = datagen.flow_from_directory(ORIG_PATH['val'],   target_size=(224, 224),
                                        batch_size=32, class_mode='categorical',
                                        shuffle=False)
test_gen  = datagen.flow_from_directory(ORIG_PATH['test'],  target_size=(224, 224),
                                        batch_size=32, class_mode='categorical',
                                        shuffle=False)

Found 20080 images belonging to 150 classes.
Found 4248 images belonging to 150 classes.
Found 4468 images belonging to 150 classes.


## 4. Build Model

> **Thay đổi chính:** Tách model thành 2 phần:
> - `feature_extractor`: EfficientNetB0 + GAP + BN + Dropout → **không có Dense cuối** → dùng để trích features sau fine-tune
> - `full_model_temp`: ghép thêm Dense 150 tạm → **chỉ dùng để fine-tune backbone** có gradient signal

In [4]:
def build_feature_extractor(input_shape=(224, 224, 3)):

    input_tensor = Input(shape=input_shape)
    backbone = EfficientNetB0(include_top=False,
                              weights='imagenet',
                              input_tensor=input_tensor)
    backbone.trainable = False  # Freeze cho giai đoạn 1

    x = backbone.output
    x = GlobalAveragePooling2D()(x)

    return Model(inputs=input_tensor, outputs=x, name='efficientnetb0_extractor')


feature_extractor = build_feature_extractor(input_shape=(224, 224, 3))

feature_extractor.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "efficientnetb0_extractor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,049,571 (15.45 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 4,049,571 (15.45 MB)

## 5. Trích xuất đặc trưng


In [6]:
def extract_features(model, generator):
    """
    Trích xuất feature vector từ GlobalAveragePooling2D.
    Returns: features (N, 1280), labels (N,)
    """
    generator.reset()
    features = model.predict(generator, verbose=1)
    labels   = generator.classes
    return features, labels

print("Đang trích xuất features từ tập train...")
X_train, y_train = extract_features(feature_extractor, train_gen)

print("Đang trích xuất features từ tập val...")
X_val, y_val = extract_features(feature_extractor, val_gen)

print("Đang trích xuất features từ tập test...")
X_test, y_test = extract_features(feature_extractor, test_gen)


print(f"\nShape features:")
print(f"  Train+Val : {X_train.shape}")
print(f"  Test      : {X_test.shape}")


Đang trích xuất features từ tập train...
628/628 ━━━━━━━━━━━━━━━━━━━━ 32s 51ms/step
Đang trích xuất features từ tập val...
133/133 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step
Đang trích xuất features từ tập test...
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 55ms/step

Shape features:
  Train+Val : (20080, 1280)
  Test      : (4468, 1280)


## 9. GridSearchCV + Random Forest

In [15]:
from sklearn.model_selection import PredefinedSplit
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_split": [2, 10],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt"],
    "bootstrap": [True]
}


X_combined = np.vstack((X_train, X_val))
y_combined = np.concatenate((y_train, y_val))

test_fold = np.concatenate([
    np.full(X_train.shape[0], -1),  # Gán toàn bộ data từ folder train thành -1
    np.full(X_val.shape[0], 0)      # Gán toàn bộ data từ folder val thành 0
])

ps = PredefinedSplit(test_fold)

rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,           # Dùng toàn bộ CPU cores
    class_weight='balanced'
)

# cv=3 để nhanh; tăng lên 5 nếu muốn kết quả ổn định hơn
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=ps,
    scoring='accuracy',
    verbose=2,
    n_jobs=1,
    refit=True           # Tự fit lại với best params trên toàn bộ X_trainval
)

print("[Random Forest] Bắt đầu GridSearchCV...")
grid_search.fit(X_combined, y_combined)

print(f"\nBest parameters : {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_*100:.2f}%")


[Random Forest] Bắt đầu GridSearchCV...

Fitting 1 folds for each of 16 candidates, totalling 16 fits

[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time= 2.7min
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time= 5.3min
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100; total time= 2.6min
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time= 5.2min
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=2, n_estimators=100; total time= 2.7min
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=2, n_estimators=200; total time= 5.4min
[CV] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=5, m